In [16]:
import numpy as np
import pandas as pd
import os
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    recall_score,
    precision_score,   
    make_scorer,
    silhouette_score,
    precision_recall_curve,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_samples
)

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestCentroid

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import time

In [17]:
# 1. CARGAR DATOS
def cargar_y_preparar_datos(ruta_archivo):
    df = pd.read_excel(ruta_archivo)
    # Filtrar solo vivienda y copiar para evitar warnings
    df_viv = df[df['Proposito'].astype(str)
                .str.contains('Vivienda', case=False, na=False)].copy()
    # Label
    df_viv['Impago_Label'] = df_viv['Impago'].map({0:0, 1:1})
    return df_viv

# Ajusta esta ruta si es necesario
ruta_real = os.path.join('..', 'Datos', 'Limpios', 'información_préstamos_limpio.xlsx')

if os.path.exists(ruta_real):
    df = cargar_y_preparar_datos(ruta_real)
else:
    print(f" ATENCIÓN: No se encuentra el archivo en {ruta_real}")
    df = pd.DataFrame() 

In [18]:
# 2. DEFINIR X e y
if not df.empty:
    target_col = "Impago_Label"
    columnas_a_eliminar = ["ID", "Impago", "Prima", "Proposito"]

    y = df[target_col]
    X = df.drop(columns=[target_col])
    X = X.drop(columns=[col for col in columnas_a_eliminar if col in X.columns])

    # Eliminar alta cardinalidad
    high_card_cols = [col for col in X.columns if X[col].nunique() > 50]
    X = X.drop(columns=high_card_cols)

    # One-hot encoding
    cat_cols = X.select_dtypes(include="object").columns
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

    X = X.astype("float32")
    
# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

In [19]:
# 4. CLUSTERING AVANZADO: TORNEO, MÉTRICAS Y VISUALIZACIÓN 3D
print("Iniciando Optimización Avanzada de Clustering ")

# 4.1. Escalado de los datos (Vital para Clustering)
scaler_cluster = StandardScaler()
X_train_cluster = scaler_cluster.fit_transform(X_train)
X_test_cluster = scaler_cluster.transform(X_test)

# 4.2. PRUEBA DE DBSCAN (Descarte por densidad)
# Lo probamos para demostrar que evaluamos algoritmos basados en densidad
print(" Probando DBSCAN (Basado en Densidad)...")
dbscan = DBSCAN(eps=2.0, min_samples=10)
db_labels = dbscan.fit_predict(X_train_cluster)
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_ruido = list(db_labels).count(-1)
print(f"DBSCAN encontró {n_clusters_db} clusters y {n_ruido} puntos de ruido (outliers).")
print("Motivo de descarte: En datos financieros continuos, DBSCAN suele agrupar casi todo en un solo clúster gigante o generar demasiado ruido. Pasamos a algoritmos particionales.\n")

# 4.3. TORNEO K-MEANS vs AGLOMERATIVO (Guardando todas las métricas)
print("Iniciando Torneo: KMeans vs Jerárquico/Aglomerativo...")
resultados_clustering = []
k_values = [2, 3, 4, 5]

for k in k_values:
    # Modelo K-Means 
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels = km.fit_predict(X_train_cluster)
    
    resultados_clustering.append({
        'Modelo': 'K-Means',
        'K': k,
        'Silhouette (↑)': silhouette_score(X_train_cluster, km_labels),
        'Calinski-Harabasz (↑)': calinski_harabasz_score(X_train_cluster, km_labels),
        'Davies-Bouldin (↓)': davies_bouldin_score(X_train_cluster, km_labels)
    })
    
    #Modelo Aglomerativo (Jerárquico)
    agg = AgglomerativeClustering(n_clusters=k)
    agg_labels = agg.fit_predict(X_train_cluster)
    
    resultados_clustering.append({
        'Modelo': 'Aglomerativo',
        'K': k,
        'Silhouette (↑)': silhouette_score(X_train_cluster, agg_labels),
        'Calinski-Harabasz (↑)': calinski_harabasz_score(X_train_cluster, agg_labels),
        'Davies-Bouldin (↓)': davies_bouldin_score(X_train_cluster, agg_labels)
    })

# Convertimos a DataFrame para ver la tabla bonita
df_metricas_clusters = pd.DataFrame(resultados_clustering)
print("TABLA COMPARATIVA DE MÉTRICAS:")
print(df_metricas_clusters.sort_values(by=['Silhouette (↑)'], ascending=False).to_string(index=False))

# 4.4. SELECCIÓN DEL GANADOR
# Buscamos la fila con el mejor Silhouette general
mejor_fila = df_metricas_clusters.loc[df_metricas_clusters['Silhouette (↑)'].idxmax()]
best_model_name = mejor_fila['Modelo']
best_k = int(mejor_fila['K'])

print(f"GANADOR DEL TORNEO: {best_model_name} con k={best_k}")

# Entrenamos el modelo ganador definitivo
if best_model_name == 'K-Means':
    best_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
else:
    best_model = AgglomerativeClustering(n_clusters=best_k)

final_labels_train = best_model.fit_predict(X_train_cluster)

# 4.5. VISUALIZACIONES AVANZADAS DEL MODELO GANADOR 
print("Generando gráfico de optimización de K...")

COLORES = {
    'Verde_Fuerte': '#74b404', 
    'Verde_Claro':  '#cdfc7d', 
    'Rojo_Corp':    '#aa044c', 
    'Morado_Os':    '#876784',
    'Morado_Cl':    '#b09eae'
}

PALETA_CLUSTERS = [
    COLORES['Verde_Fuerte'],
    COLORES['Rojo_Corp'],
    COLORES['Morado_Os'],
    COLORES['Morado_Cl'],
    COLORES['Verde_Claro']
]

# Extraemos los datos de la tabla de métricas que ya calculamos (solo de KMeans para el Codo)
df_kmeans = df_metricas_clusters[df_metricas_clusters['Modelo'] == 'K-Means'].sort_values('K')
k_vals = df_kmeans['K'].tolist()
sil_vals = df_kmeans['Silhouette (↑)'].tolist()

inercias = []
for k in k_vals:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_train_cluster)
    inercias.append(km_temp.inertia_)

fig_optim = go.Figure()

fig_optim.add_trace(go.Scatter(
    x=k_vals, y=inercias, mode='lines+markers',
    name='Inercia (Codo)',
    line=dict(color=COLORES['Morado_Os'], width=3),
    marker=dict(size=10, color=COLORES['Morado_Os'])
))

fig_optim.add_trace(go.Scatter(
    x=k_vals, y=sil_vals, mode='lines+markers',
    name='Silhouette Score',
    yaxis='y2',
    line=dict(color=COLORES['Rojo_Corp'], width=3, dash='dot'),
    marker=dict(size=10, symbol='diamond', color=COLORES['Rojo_Corp'])
))

fig_optim.update_layout(
    title='Método del Codo y Coeficiente de Silueta por K',
    xaxis=dict(title='Número de Clusters (k)', tickvals=k_vals),
    yaxis=dict(
        title=dict(text='Inercia (Compactación)', font=dict(color=COLORES['Morado_Os'])),
        tickfont=dict(color=COLORES['Morado_Os'])
    ),
    yaxis2=dict(
        title=dict(text='Silhouette (Separación)', font=dict(color=COLORES['Rojo_Corp'])),
        tickfont=dict(color=COLORES['Rojo_Corp']),
        overlaying='y',
        side='right'
    ),
    template="plotly_white",
    legend=dict(x=0.01, y=0.99, bordercolor="Black", borderwidth=1)
)

fig_optim.add_vline(
    x=best_k,
    line_width=2,
    line_dash="dash",
    line_color=COLORES['Verde_Fuerte']
)
fig_optim.add_annotation(x=best_k, y=max(inercias), text=f"K Óptimo = {best_k}", showarrow=True, arrowhead=1)

fig_optim.show()

print("Generando Gráfico de Silueta interactivo para el modelo ganador...")

silhouette_vals = silhouette_samples(X_train_cluster, final_labels_train)
avg_score = mejor_fila['Silhouette (↑)']

fig_silueta = go.Figure()

y_lower = 10
colores_clusters = PALETA_CLUSTERS

for i in range(best_k):
    ith_cluster_vals = silhouette_vals[final_labels_train == i]
    ith_cluster_vals.sort()
    
    size_cluster_i = ith_cluster_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    
    color = colores_clusters[i % len(colores_clusters)]
    
    fig_silueta.add_trace(go.Scatter(
        x=np.concatenate(([0], ith_cluster_vals, [0])), 
        y=np.concatenate(([y_lower], np.arange(y_lower, y_upper), [y_upper])),
        fill='toself',
        fillcolor=color,
        line=dict(color=color, width=0),
        name=f'Cluster {i} (N={size_cluster_i})',
        hoverinfo='x+name'
    ))
    
    fig_silueta.add_annotation(
        x=-0.05,
        y=y_lower + 0.5 * size_cluster_i,
        text=str(i),
        showarrow=False,
        font=dict(size=14, color="black"),
        xanchor="right"
    )
    
    y_lower = y_upper + 10  


fig_silueta.add_vline(
    x=avg_score,
    line_width=2,
    line_dash="dash",
    line_color=COLORES['Rojo_Corp'],
    annotation_text=f"Promedio: {avg_score:.3f}"
)

fig_silueta.update_layout(
    title=f"Gráfico de Silueta Interactivo - {best_model_name} (k={best_k})",
    xaxis_title="Coeficiente de Silueta",
    yaxis_title="Etiqueta del Cluster",
    yaxis=dict(showticklabels=False, range=[0, y_upper + 10]), 
    template="plotly_white",
    hovermode="y unified"
)
fig_silueta.show()

# REDUCCIÓN PCA A 3 DIMENSIONES 
print("Generando Visualizaciones 2D y 3D (PCA)...")
pca_3d = PCA(n_components=3, random_state=42)
X_pca_3d = pca_3d.fit_transform(X_train_cluster)

df_pca = pd.DataFrame(data=X_pca_3d, columns=['PCA1', 'PCA2', 'PCA3'])
df_pca['Cluster'] = final_labels_train.astype(str)

#C) GRÁFICO 2D 
fig_2d = px.scatter(
    df_pca,
    x='PCA1',
    y='PCA2',
    color='Cluster',
    title=f'Visualización 2D de Clusters ({best_model_name})',
    color_discrete_sequence=PALETA_CLUSTERS,
    opacity=0.6
)
fig_2d.update_layout(template="plotly_white")
fig_2d.show()

# GRÁFICO 3D INTERACTIVO
fig_3d = px.scatter_3d(
    df_pca,
    x='PCA1',
    y='PCA2',
    z='PCA3',
    color='Cluster',
    title=f'Visualización 3D de Clusters ({best_model_name})',
    color_discrete_sequence=PALETA_CLUSTERS,
    opacity=0.7
)
fig_3d.update_layout(scene=dict(xaxis_title='PCA 1', yaxis_title='PCA 2', zaxis_title='PCA 3'),
                     margin=dict(l=0, r=0, b=0, t=40))
fig_3d.show()


# INYECCIÓN SEGURA EN TRAIN Y TEST (Para el Modelo Predictivo)
if hasattr(best_model, "predict"):
    final_labels_test = best_model.predict(X_test_cluster)
else:
    centroid_clf = NearestCentroid()
    centroid_clf.fit(X_train_cluster, final_labels_train)
    final_labels_test = centroid_clf.predict(X_test_cluster)

# One-Hot Encoding
train_dummies = pd.get_dummies(final_labels_train, prefix='Cluster_Group')
test_dummies = pd.get_dummies(final_labels_test, prefix='Cluster_Group')

# Alinear columnas por si el test no tiene algún cluster
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

# Unir a los datos originales
train_dummies.index = X_train.index
test_dummies.index = X_test.index
X_train = pd.concat([X_train, train_dummies], axis=1)
X_test = pd.concat([X_test, test_dummies], axis=1)

print(f"Fusión completada. Variables de cluster añadidas. Nuevas columnas: {list(train_dummies.columns)}")

Iniciando Optimización Avanzada de Clustering 
 Probando DBSCAN (Basado en Densidad)...
DBSCAN encontró 157 clusters y 904 puntos de ruido (outliers).
Motivo de descarte: En datos financieros continuos, DBSCAN suele agrupar casi todo en un solo clúster gigante o generar demasiado ruido. Pasamos a algoritmos particionales.

Iniciando Torneo: KMeans vs Jerárquico/Aglomerativo...
TABLA COMPARATIVA DE MÉTRICAS:
      Modelo  K  Silhouette (↑)  Calinski-Harabasz (↑)  Davies-Bouldin (↓)
Aglomerativo  4        0.180066            1092.192085            1.859031
Aglomerativo  3        0.160018            1100.259251            1.973671
     K-Means  4        0.150470             987.581190            2.094499
     K-Means  2        0.146857            1299.457049            2.453518
     K-Means  5        0.145566             996.090434            1.898896
Aglomerativo  5        0.142823             988.774579            1.907550
Aglomerativo  2        0.141844            1231.484449          

Generando Gráfico de Silueta interactivo para el modelo ganador...


Generando Visualizaciones 2D y 3D (PCA)...


Fusión completada. Variables de cluster añadidas. Nuevas columnas: ['Cluster_Group_0', 'Cluster_Group_1', 'Cluster_Group_2', 'Cluster_Group_3']


In [20]:
# 4.4.1. MÉTRICAS AVANZADAS: COHESIÓN (INTRA) Y SEPARACIÓN (INTER)
from scipy.spatial.distance import cdist, pdist

print("Calculando métricas internas de distancias del modelo ganador")

# 1. Calculamos los centroides reales de cada cluster
centroides = []
for i in range(best_k):
    puntos_cluster = X_train_cluster[final_labels_train == i]
    centroide = puntos_cluster.mean(axis=0)
    centroides.append(centroide)

centroides = np.array(centroides)

# 2. Distancia INTER-cluster
distancias_centroides = pdist(centroides, metric='euclidean')
separacion_media = np.mean(distancias_centroides)

print(f"Distancia Media INTER-Cluster (Separación): {separacion_media:.4f} (↑ Cuanto mayor, más distintos son los perfiles)")

# 3. Distancia INTRA-cluster
cohesion_global = []

print("Detalle de Cohesión INTRA-Cluster por grupo:")

for i in range(best_k):

    puntos_cluster = X_train_cluster[final_labels_train == i]

    dist_al_centroide = cdist(
        puntos_cluster,
        [centroides[i]],
        metric='euclidean'
    )

    cohesion_cluster = np.mean(dist_al_centroide)

    cohesion_global.append(cohesion_cluster)

    n_puntos = len(puntos_cluster)

    print(
        f"   -> Cluster {i} (N={n_puntos} clientes): "
        f"Distancia media = {cohesion_cluster:.4f} "
        f"(↓ Menor es más compacto)"
    )

# Cohesión media global
cohesion_media_global = np.mean(cohesion_global)

print(
    f"Distancia Media INTRA-Cluster (Cohesión general): "
    f"{cohesion_media_global:.4f}"
)

# Ratio final
ratio_cs = cohesion_media_global / separacion_media

print(
    f"Ratio Cohesión / Separación: {ratio_cs:.4f} "
    f"(↓ Buscamos un valor bajo)"
)


# 4.4.2. VISUALIZACIÓN DE COHESIÓN Y SEPARACIÓN (PLOTLY)

print("Generando visualizaciones de métricas internas...")

# 1. BOXPLOT DE COHESIÓN
datos_boxplot = []

for i in range(best_k):

    puntos_cluster = X_train_cluster[final_labels_train == i]

    dist_al_centroide = cdist(
        puntos_cluster,
        [centroides[i]],
        metric='euclidean'
    ).flatten()

    temp_df = pd.DataFrame({
        'Distancia al Centroide': dist_al_centroide,
        'Cluster': f'Cluster {i}'
    })

    datos_boxplot.append(temp_df)

df_boxplot = pd.concat(datos_boxplot)

fig_cohesion = px.box(
    df_boxplot,
    x='Cluster',
    y='Distancia al Centroide',
    color='Cluster',
    title='Distribución de la Cohesión Intra-Cluster (Menor es más compacto)',
    points="outliers",
    color_discrete_sequence=PALETA_CLUSTERS
)

fig_cohesion.update_layout(
    template="plotly_white",
    showlegend=False
)

fig_cohesion.show()


# 2. HEATMAP DE SEPARACIÓN
from scipy.spatial.distance import squareform

matriz_distancias = squareform(distancias_centroides)

nombres_clusters = [f'Cluster {i}' for i in range(best_k)]

fig_separacion = px.imshow(
    matriz_distancias,
    text_auto=".2f",
    x=nombres_clusters,
    y=nombres_clusters,
    color_continuous_scale=[
        COLORES['Verde_Claro'],
        COLORES['Morado_Cl'],
        COLORES['Morado_Os'],
        COLORES['Rojo_Corp']
    ],
    title='Matriz de Separación Inter-Cluster (Distancia entre Centroides)'
)

fig_separacion.update_layout(
    xaxis_title="Cluster",
    yaxis_title="Cluster",
    template="plotly_white"
)

fig_separacion.show()

Calculando métricas internas de distancias del modelo ganador
Distancia Media INTER-Cluster (Separación): 4.3686 (↑ Cuanto mayor, más distintos son los perfiles)
Detalle de Cohesión INTRA-Cluster por grupo:
   -> Cluster 0 (N=3853 clientes): Distancia media = 3.1045 (↓ Menor es más compacto)
   -> Cluster 1 (N=3071 clientes): Distancia media = 3.1196 (↓ Menor es más compacto)
   -> Cluster 2 (N=443 clientes): Distancia media = 3.3082 (↓ Menor es más compacto)
   -> Cluster 3 (N=826 clientes): Distancia media = 3.2068 (↓ Menor es más compacto)
Distancia Media INTRA-Cluster (Cohesión general): 3.1848
Ratio Cohesión / Separación: 0.7290 (↓ Buscamos un valor bajo)
Generando visualizaciones de métricas internas...


In [22]:
# EXTRA: PERFILADO DE CLUSTERS (Que tipo de personas hay en cada cluster)
print("RADIOGRAFÍA DE LOS 4 CLUSTERS")

# Juntamos los datos temporalmente para analizarlos
df_perfil = X_train.copy()
df_perfil['Impago_Real'] = y_train
df_perfil['Cluster_Asignado'] = final_labels_train  # Del 0 al 3

# Vemos cuánta gente hay y cuántos morosos tiene cada cluster
resumen_clusters = df_perfil.groupby('Cluster_Asignado').agg(
    Total_Clientes=('Impago_Real', 'count'),
    Tasa_Morosidad_Pct=('Impago_Real', lambda x: (x.mean() * 100).round(2))
).reset_index()

print("MOROSIDAD POR CLUSTER:")
print(resumen_clusters.to_string(index=False))

# Vemos cómo es el cliente medio de cada cluster
# Quitamos las columnas dummy de los propios clusters para no ensuciar
cols_a_ignorar = [c for c in df_perfil.columns if "Cluster_Group" in c]
df_perfil_limpio = df_perfil.drop(columns=cols_a_ignorar)

perfil_variables = df_perfil_limpio.groupby('Cluster_Asignado').mean().round(2).T

print("PERFIL MEDIO DEL CLIENTE EN CADA CLUSTER:")
# Mostramos todas las filas para que puedas ver todas las variables
pd.set_option('display.max_rows', None) 
print(perfil_variables)

RADIOGRAFÍA DE LOS 4 CLUSTERS
MOROSIDAD POR CLUSTER:
 Cluster_Asignado  Total_Clientes  Tasa_Morosidad_Pct
                0            3853               11.06
                1            3071               14.00
                2             443                9.03
                3             826                4.96
PERFIL MEDIO DEL CLIENTE EN CADA CLUSTER:
Cluster_Asignado                               0          1          2  \
Num_Creditos                            2.210000   2.200000   2.010000   
Duracion                               32.549999  33.279999  33.099998   
Ratio_Deuda_Ingresos                    0.510000   0.490000   0.450000   
Posesion_Hipoteca                       0.420000   0.430000   0.420000   
Personas_Cargo                          0.460000   0.460000   0.380000   
Fiador                                  0.510000   0.500000   0.490000   
Estudios_Escolar                        0.990000   0.000000   0.000000   
Estudios_Grado Universitario            0.0

🔴 Cluster 1 – “Universitarios jóvenes de mayor riesgo”

- 99% con Grado universitario
- 73% solteros
- Ratio deuda/ingresos: 0,49
- Morosidad: 14% (la más alta)

Interpretación:

- Perfil joven, posiblemente en primeras etapas laborales.
- Aunque tienen formación superior, pueden tener inestabilidad laboral o ingresos aún bajos.
- Alta proporción de solteros → menor red de apoyo financiero.

Este grupo representa riesgo estructural pese a buena educación.

🟠 Cluster 0 – “Perfil escolar tradicional con riesgo medio”

- 99% estudios escolares (sin grado universitario)
- 54% solteros
- Ratio deuda/ingresos más alto (0,51)
- Morosidad: 11%

Interpretación:

- Nivel educativo más bajo.
- Mayor presión financiera relativa.
- Riesgo moderado, probablemente más sensible a shocks económicos.

Segmento numeroso que aporta volumen de mora.

🟡 Cluster 2 – “Perfil Máster relativamente estable”

- 100% con Máster
- Ratio deuda/ingresos más bajo (0,45)
- 9% morosidad

Interpretación:

- Mayor cualificación → mejor empleabilidad.
- Mejor equilibrio deuda/ingresos.
- Aunque no es el más seguro, presenta riesgo controlado.

Podría ser segmento interesante para productos premium.

🟢 Cluster 3 – “Divorciados maduros y más responsables”

- 100% divorciados
- 0% solteros
- Morosidad: 4,96% (la más baja)

Interpretación:

- Perfil más maduro.
- Posiblemente mayor estabilidad económica y experiencia crediticia.
- Aunque no necesariamente con mayor educación, muestran mejor comportamiento de pago.

Segmento más rentable y estable.